# Binary-Sequence Prediction with a Transformer

We examine whether a binary sequence contains patterns that can be learned and used to predict future bits. For a sequence intended to be random, consistently better-than-chance prediction would indicate that the sequence contains detectable structure.

Given a binary string, the data is split chronologically into training, validation, and test sets. Rolling windows of the previous `context_len` bits are used to predict the following bit at each position. A simple causal Transformer is trained on the training set, selected using validation loss, and evaluated on test data.

In [ ]:
# --- Packages ---

import copy
import math
import random
from typing import Dict, Any

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

## Setup

In [ ]:
# --- Setup ---

def set_seed(seed: int = 42):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def string_to_tensor(s: str) -> torch.Tensor:
    s = s.strip()

    if not s:
        raise ValueError("binary_string is empty.")
    if any(c not in "01" for c in s):
        raise ValueError("binary_string must contain only 0 and 1.")

    return torch.tensor([int(c) for c in s], dtype=torch.int64)

def tensor_to_string(x: torch.Tensor) -> str:
    return "".join(str(int(v)) for v in x)

# --- Dataset ---

def make_windows(bits: torch.Tensor, context_len: int):
    """
    Create examples of the form [context_len bits] -> [same window shifted one bit ahead],
    so every position is trained to predict the bit that follows it.
    """
    if len(bits) <= context_len:
        raise ValueError(f"Need more than {context_len} bits in this split.")

    X, y = [], []

    # Rolling context window
    for i in range(context_len, len(bits)):
        X.append(bits[i - context_len:i])
        y.append(bits[i - context_len + 1:i + 1]) # next bit for every position

    X, y = torch.stack(X), torch.stack(y)

    return TensorDataset(X, y)

## Transformer

A causal transformer maps each binary context to predictions for the following bits. The causal mask prevents a position from using future bits when making its prediction.

In [ ]:
# --- Transformer ---

class BinaryTransformer(nn.Module):

    def __init__(
        self,
        context_len=64,
        d_model=64,
        nhead=4,
        num_layers=2,
        dropout=0.1
    ):
        super().__init__()

        if d_model % nhead != 0:
            raise ValueError("d_model must be divisible by nhead.")

        self.context_len = context_len

        # Token embedding
        self.token_embedding = nn.Embedding(2, d_model)
        nn.init.normal_(self.token_embedding.weight, std=0.02)

        # Positional embedding
        self.position_embedding = nn.Parameter(torch.randn(1, context_len, d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            activation="gelu",
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.norm = nn.LayerNorm(d_model)
        self.output = nn.Linear(d_model, 2)

        # Causal mask
        mask = torch.triu(torch.ones(context_len, context_len, dtype=torch.bool), diagonal=1)
        self.register_buffer("causal_mask", mask)

    def forward(self, x):
        # x.shape = [batch, context_len] -> output shape = [batch, context_len, 2]
        T = x.shape[1]

        h = self.token_embedding(x)
        h = h + self.position_embedding[:, :T, :]
        h = self.transformer(h, mask=self.causal_mask[:T, :T])
        h = self.norm(h)

        return self.output(h)

In [ ]:
# --- Metrics ---

@torch.no_grad()
def evaluate(
    model,
    dataset,
    device,
    batch_size=256
) -> Dict[str, float]:

    model.eval()

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False
    )

    loss_fn = nn.CrossEntropyLoss(reduction="sum")

    total_loss = 0.0
    total_correct = 0
    total = 0

    all_positions_loss = 0.0
    all_positions_total = 0

    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)                 # [batch, T, 2]
        last_logits = logits[:, -1, :]    # [batch, 2]
        last_y = y[:, -1]                 # [batch]

        total_loss += loss_fn(last_logits, last_y).item()
        total_correct += (last_logits.argmax(dim=1) == last_y).sum().item()
        total += len(last_y)

        all_positions_loss += loss_fn(logits.reshape(-1, 2), y.reshape(-1)).item()
        all_positions_total += y.numel()

    return {
        "cross_entropy": total_loss / total,
        "accuracy": total_correct / total,
        "all_positions_cross_entropy": all_positions_loss / all_positions_total,
    }

## Predictor

The trained model can generate bits autoregressively by repeatedly using the most recent context window to predict the next bit.


In [ ]:
# --- Autoregressive prediction ---

@torch.no_grad()
def predict_next_n(
    model,
    binary_string: str,
    n: int,
    device
) -> str:

    model.eval()

    history = string_to_tensor(binary_string).to(device)

    if len(history) < model.context_len:
        raise ValueError("binary_string must contain at least context_len bits.")

    predictions = []

    for _ in range(n):
        context = history[-model.context_len:]
        context = context.unsqueeze(0)
        logits = model(context)[:, -1, :]   # only the prediction after the window
        next_bit = int(logits.argmax(dim=1).item())
        predictions.append(str(next_bit))
        history = torch.cat([history, torch.tensor([next_bit], device=device)])

    return "".join(predictions)

## Train

The training routine constructs the chronological data splits, trains the transformer with early stopping, evaluates one-step and multi-step prediction accuracy, and generates forecasts.


In [ ]:
# --- Main function ---

def train_binary_transformer(
    binary_string: str,
    n: int = 16,
    context_len: int = 64,
    train_fraction: float = 0.70,
    validation_fraction: float = 0.15,
    d_model: int = 64,
    nhead: int = 4,
    num_layers: int = 2,
    dropout: float = 0.1,
    learning_rate: float = 1e-3,
    weight_decay: float = 0.01,
    batch_size: int = 128,
    max_epochs: int = 100,
    patience: int = 10,
    seed: int = 42
) -> Dict[str, Any]:

    set_seed(seed)

    bits = string_to_tensor(binary_string)
    N = len(bits)

    # Chronological train/validation/test split
    train_end = int(train_fraction * N)
    validation_end = train_end + int(validation_fraction * N)

    train_bits = bits[:train_end]
    validation_bits = bits[train_end:validation_end]
    test_bits = bits[validation_end:]

    if min(
        len(train_bits),
        len(validation_bits),
        len(test_bits)
    ) <= context_len:

        raise ValueError(
            "Each split must contain more than context_len bits. "
            "Use a smaller context_len or a longer sequence."
        )

    # Datasets
    train_dataset = make_windows(train_bits, context_len)
    validation_dataset = make_windows(validation_bits, context_len)
    test_dataset = make_windows(test_bits, context_len)

    # Model
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = BinaryTransformer(
        context_len=context_len,
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        dropout=dropout
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    loss_fn = nn.CrossEntropyLoss()

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )

    # Training + early stopping
    best_validation_loss = math.inf
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(1, max_epochs + 1):
        model.train()

        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(X)
            loss = loss_fn(logits.reshape(-1, 2), y.reshape(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        validation_metrics = evaluate(
            model,
            validation_dataset,
            device,
            batch_size,
        )

        validation_loss = validation_metrics["cross_entropy"]

        if validation_loss < best_validation_loss:
            best_validation_loss = validation_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0

        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            break

    # Restore best model
    if best_state is not None:
        model.load_state_dict(best_state)

    # Metrics
    train_metrics = evaluate(
        model,
        train_dataset,
        device,
        batch_size,
    )

    validation_metrics = evaluate(
        model,
        validation_dataset,
        device,
        batch_size,
    )

    test_metrics = evaluate(
        model,
        test_dataset,
        device,
        batch_size,
    )

    # Predict first n test bits autoregressively
    # Use all information available before the test set
    history_before_test = tensor_to_string(torch.cat([train_bits, validation_bits]))

    n_test = min(n, len(test_bits))
    test_prediction = predict_next_n(
        model,
        history_before_test,
        n_test,
        device,
    )

    actual_test_bits = tensor_to_string(test_bits[:n_test])

    # Metrics for the genuine n-step forecast
    forecast_correct = sum(
        a == b
        for a, b in zip(
            test_prediction,
            actual_test_bits,
        )
    )

    forecast_accuracy = (forecast_correct / n_test)
    exact_match = (test_prediction == actual_test_bits)

    # Predict n bits after the entire supplied string
    future_prediction = predict_next_n(
        model,
        binary_string,
        n,
        device,
    )

    return {
        "model": model,
        "device": str(device),
        "epochs_trained": epoch,
        "split_sizes": {
            "train": len(train_bits),
            "validation": len(validation_bits),
            "test": len(test_bits),
        },
        "metrics": {
            "train": train_metrics,
            "validation": validation_metrics,
            "test_one_step": test_metrics,
            "test_n_step_accuracy": forecast_accuracy,
            "test_n_step_exact_match": float(exact_match),
        },
        "test_n_step_prediction": test_prediction,
        "test_n_step_actual": actual_test_bits,
        # This has no metric because the true future is unknown:
        "next_n_after_full_string": future_prediction
    }

### RNG Example

A binary sequence generated from a random number generator provides a baseline for a sequence with no intended structure.


In [ ]:
# --- Example ---

if __name__ == "__main__":
    number = random.getrandbits(2000)
    binary_string = format(number, '02000b')  # pad so leading zeros aren't dropped

    result = train_binary_transformer(
        binary_string,
        n=8,
        context_len=16,
        max_epochs=50,
    )

    print("Device:", result["device"])
    print("Split sizes:", result["split_sizes"])
    print("Epochs trained:", result["epochs_trained"])

    print("\nMetrics:")
    for name, metrics in result["metrics"].items():
        print(f"{name}: {metrics}")

    print("\nFirst n test bits:")
    print("Predicted:", result["test_n_step_prediction"])
    print("Actual:   ", result["test_n_step_actual"])

    print("\nPredicted next n bits after the entire string:")
    print(result["next_n_after_full_string"])

Device: cpu
Split sizes: {'train': 1400, 'validation': 300, 'test': 300}
Epochs trained: 17

Metrics:
train: {'cross_entropy': 0.6937387775134489, 'accuracy': 0.4920520231213873, 'all_positions_cross_entropy': 0.6939669261778021}
validation: {'cross_entropy': 0.6906460976936448, 'accuracy': 0.5387323943661971, 'all_positions_cross_entropy': 0.6900887220678195}
test_one_step: {'cross_entropy': 0.6923911336442115, 'accuracy': 0.5176056338028169, 'all_positions_cross_entropy': 0.692939174007362}
test_n_step_accuracy: 0.25
test_n_step_exact_match: 0.0

First n test bits:
Predicted: 00000000
Actual:    11111010

Predicted next n bits after the entire string:
00000000


### Sanity-Check Example

A deterministic repeating sequence is used to verify that the model can learn a simple, strongly predictable pattern.


In [ ]:
# --- Example ---

if __name__ == "__main__":
    binary_string = (
        "00101101001011010010110100101101"
        "00101101001011010010110100101101"
        "00101101001011010010110100101101"
        "00101101001011010010110100101101"
        "00101101001011010010110100101101"
        "00101101001011010010110100101101"
        "00101101001011010010110100101101"
    )

    result = train_binary_transformer(
        binary_string,
        n=8,
        context_len=16,
        max_epochs=30,
    )

    print("Device:", result["device"])
    print("Split sizes:", result["split_sizes"])
    print("Epochs trained:", result["epochs_trained"])

    print("\nMetrics:")
    for name, metrics in result["metrics"].items():
        print(f"{name}: {metrics}")

    print("\nFirst n test bits:")
    print("Predicted:", result["test_n_step_prediction"])
    print("Actual:   ", result["test_n_step_actual"])

    print("\nPredicted next n bits after the entire string:")
    print(result["next_n_after_full_string"])

Device: cpu
Split sizes: {'train': 156, 'validation': 33, 'test': 35}
Epochs trained: 30

Metrics:
train: {'cross_entropy': 0.0021291187166103294, 'accuracy': 1.0, 'all_positions_cross_entropy': 0.09255398937634059}
validation: {'cross_entropy': 0.0020972459631807662, 'accuracy': 1.0, 'all_positions_cross_entropy': 0.0931263250463149}
test_one_step: {'cross_entropy': 0.0020878875726147703, 'accuracy': 1.0, 'all_positions_cross_entropy': 0.09051976078434994}
test_n_step_accuracy: 1.0
test_n_step_exact_match: 1.0

First n test bits:
Predicted: 10100101
Actual:    10100101

Predicted next n bits after the entire string:
00101101


### 90% Example

A repeating pattern with 10% of its bits randomly flipped tests prediction when the underlying pattern is mostly preserved.

In [ ]:
# --- Example ---

if __name__ == "__main__":
    random.seed(0)
    pattern = "00101101" * 200

    binary_string = "".join(
        bit if random.random() > 0.1 else str(1 - int(bit))  # flip 10% of bits
        for bit in pattern
    )

    result = train_binary_transformer(
        binary_string,
        n=8,
        context_len=16,
        max_epochs=50,
    )

    print("Device:", result["device"])
    print("Split sizes:", result["split_sizes"])
    print("Epochs trained:", result["epochs_trained"])

    print("\nMetrics:")
    for name, metrics in result["metrics"].items():
        print(f"{name}: {metrics}")

    print("\nFirst n test bits:")
    print("Predicted:", result["test_n_step_prediction"])
    print("Actual:   ", result["test_n_step_actual"])

    print("\nPredicted next n bits after the entire string:")
    print(result["next_n_after_full_string"])

Device: cpu
Split sizes: {'train': 1120, 'validation': 240, 'test': 240}
Epochs trained: 25

Metrics:
train: {'cross_entropy': 0.3741270873857581, 'accuracy': 0.8713768115942029, 'all_positions_cross_entropy': 0.45168879757756775}
validation: {'cross_entropy': 0.4049203906740461, 'accuracy': 0.84375, 'all_positions_cross_entropy': 0.5087737696511405}
test_one_step: {'cross_entropy': 0.3185147983687265, 'accuracy': 0.9017857142857143, 'all_positions_cross_entropy': 0.4066362040383475}
test_n_step_accuracy: 1.0
test_n_step_exact_match: 1.0

First n test bits:
Predicted: 00101101
Actual:    00101101

Predicted next n bits after the entire string:
00101101


### Me Example

A manually generated binary sequence is used to test the model.


In [ ]:
# --- Example ---

if __name__ == "__main__":
    binary_string = ("010010110101011101001011010110100101101001111010100101101001110101011010001110100101001010110100101101010101101011101011010010111010100101001")

    result = train_binary_transformer(
        binary_string,
        n=8,
        context_len=16,
        max_epochs=50,
    )

    print("Device:", result["device"])
    print("Split sizes:", result["split_sizes"])
    print("Epochs trained:", result["epochs_trained"])

    print("\nMetrics:")
    for name, metrics in result["metrics"].items():
        print(f"{name}: {metrics}")

    print("\nFirst n test bits:")
    print("Predicted:", result["test_n_step_prediction"])
    print("Actual:   ", result["test_n_step_actual"])

    print("\nPredicted next n bits after the entire string:")
    print(result["next_n_after_full_string"])

Device: cpu
Split sizes: {'train': 98, 'validation': 21, 'test': 22}
Epochs trained: 12

Metrics:
train: {'cross_entropy': 0.6052713161561547, 'accuracy': 0.7195121951219512, 'all_positions_cross_entropy': 0.6228290650902725}
validation: {'cross_entropy': 0.4554177761077881, 'accuracy': 0.8, 'all_positions_cross_entropy': 0.45011072158813475}
test_one_step: {'cross_entropy': 0.5203880469004313, 'accuracy': 0.8333333333333334, 'all_positions_cross_entropy': 0.6127705574035645}
test_n_step_accuracy: 0.5
test_n_step_exact_match: 0.0

First n test bits:
Predicted: 01010101
Actual:    10100101

Predicted next n bits after the entire string:
01010101


### Live Demonstration

The trained model predicts each new bit as it is entered. With online learning enabled, the model also updates itself using the sequence as it is typed.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

model = result["model"] # trained model
device = next(model.parameters()).device
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
loss_fn = nn.CrossEntropyLoss()

ONLINE_LEARNING = True # keep training on what you type
SHOW_GUESS = False # showing the guess lets you deliberately avoid it

state = {"typed": "", "guess": None, "conf": None, "correct": 0, "total": 0}

box = widgets.Text(placeholder="Type 0s and 1s...", layout=widgets.Layout(width="100%"))
status = widgets.HTML()
display(box, status)

def predict_next(bits):
    model.eval()
    with torch.no_grad():
        ctx = string_to_tensor(bits[-model.context_len:]).unsqueeze(0).to(device)
        probs = torch.softmax(model(ctx)[:, -1, :], dim=1)[0]

    return int(probs.argmax()), float(probs.max())

def learn_from(bits):
    model.train()
    X = string_to_tensor(bits[-model.context_len - 1:-1]).unsqueeze(0).to(device)
    y = string_to_tensor(bits[-model.context_len:]).unsqueeze(0).to(device)
    optimizer.zero_grad()
    loss_fn(model(X).reshape(-1, 2), y.reshape(-1)).backward()
    optimizer.step()

def on_change(change):
    new = "".join(c for c in change["new"] if c in "01")

    if len(new) < len(state["typed"]): # handle deletions: resync, drop stale guess
        state["typed"], state["guess"] = new, None
    else:
        for bit in new[len(state["typed"]):]:
            if state["guess"] is not None: # score the guess made before this bit
                state["correct"] += (state["guess"] == int(bit))
                state["total"] += 1
            state["typed"] += bit

            if ONLINE_LEARNING and len(state["typed"]) > model.context_len:
                learn_from(state["typed"])
            if len(state["typed"]) >= model.context_len:
                state["guess"], state["conf"] = predict_next(state["typed"])

    need = model.context_len - len(state["typed"])
    if need > 0:
        status.value = f"Type {need} more bits to start..."
        return

    acc = state["correct"] / state["total"] if state["total"] else 0
    guess = f"Next guess: <b>{state['guess']}</b> ({state['conf']:.0%}) · " if SHOW_GUESS else ""
    status.value = f"{guess}Model score: {state['correct']}/{state['total']} ({acc:.0%})"

box.observe(on_change, names="value")
on_change({"new": ""})

Text(value='', layout=Layout(width='100%'), placeholder='Type 0s and 1s...')

HTML(value='')